In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_8.py ──
"""
Shared infrastructure for MLFP02 Exercise 8 — FeatureStore + Feature Engineering.

Contains: HDB resale data loading, FeatureStore / ExperimentTracker setup
through kailash-ml, and OLS-from-scratch helpers reused across the four
R10 technique files:

    01_feature_schema.py        — FeatureSchema v1 + validation
    02_point_in_time.py         — Leakage prevention + temporal correctness
    03_rolling_features.py      — FeatureSchema v2 + group_by_dynamic
    04_modeling_with_features.py — Regression + hypothesis tests + Bayes

Technique-specific logic (schema construction, rolling window design,
coefficient interpretation) belongs in the per-technique files. This
module only owns infrastructure and reusable numeric helpers.
"""

from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_STORE_URL = "sqlite:///mlfp02_ex8_features.db"
FEATURE_TABLE_PREFIX = "kml_feat_"
EXPERIMENT_NAME = "mlfp02_ex8_hdb_features"


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (data.gov.sg)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_resale() -> pl.DataFrame:
    """Load HDB resale transactions with a parsed transaction_date column.

    The raw file stores `month` as "YYYY-MM"; we convert it to a polars
    Date so every downstream technique can sort, filter, and roll on a
    real temporal axis without string parsing.
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date")
    )
    return hdb


# ════════════════════════════════════════════════════════════════════════
# FEATURE STORE + EXPERIMENT TRACKER — kailash-ml wiring
# ════════════════════════════════════════════════════════════════════════


async def setup_feature_store() -> tuple[Any, Any, Any, bool]:
    """Create (conn, FeatureStore, ExperimentTracker, has_backend) for kailash-ml 1.1.1.

    Returns ``has_backend=False`` if the infrastructure is unavailable.
    Callers handle the degraded path by running the Polars-only versions
    of each operation.

    Note: the first tuple element is now a ``ConnectionManager`` rather than
    the old ``StoreFactory`` — kailash-ml's ExperimentTracker no longer
    accepts a positional store object; it constructs its own through the
    ``store_url`` factory. We still return a ConnectionManager so FeatureStore
    has the connection it needs.
    """
    try:
        from kailash.db import ConnectionManager
        from kailash_ml import ExperimentTracker, FeatureStore

        conn = ConnectionManager(FEATURE_STORE_URL)
        await conn.initialize()
        fs = FeatureStore(conn, table_prefix=FEATURE_TABLE_PREFIX)
        tracker = await ExperimentTracker.create(store_url=FEATURE_STORE_URL)
        return conn, fs, tracker, True
    except Exception as exc:  # noqa: BLE001 — degrade gracefully
        print(
            f"  [warn] FeatureStore backend unavailable "
            f"({type(exc).__name__}: {exc})"
        )
        return None, None, None, False


# ════════════════════════════════════════════════════════════════════════
# FEATURE COMPUTATION — v1 (basic property) and v2 (rolling market)
# ════════════════════════════════════════════════════════════════════════


def compute_v1_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute version-1 HDB property features from raw transactions.

    Produces: storey_midpoint, price_per_sqm, remaining_lease_years,
    transaction_id (row index). These are the base features v2 extends.
    """
    return df.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    ).with_row_index("transaction_id")


def compute_v2_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute v2 features = v1 + rolling town-level market context.

    Uses polars ``group_by_dynamic`` on ``transaction_date`` bucketed by
    month, then a 6-month rolling window per town. The first six months
    per town have nulls (warm-up period) — callers must ``drop_nulls``
    before modelling.
    """
    result = compute_v1_features(df).sort("transaction_date")

    town_stats = (
        result.group_by_dynamic("transaction_date", every="1mo", group_by="town")
        .agg(
            pl.col("resale_price").median().alias("monthly_median"),
            pl.col("resale_price").count().alias("monthly_volume"),
        )
        .sort("town", "transaction_date")
    )

    town_stats = town_stats.with_columns(
        pl.col("monthly_median")
        .rolling_mean(window_size=6)
        .over("town")
        .alias("town_median_price"),
        pl.col("monthly_volume")
        .rolling_sum(window_size=6)
        .over("town")
        .alias("town_transaction_volume"),
        (
            (pl.col("monthly_median") - pl.col("monthly_median").shift(6).over("town"))
            / pl.col("monthly_median").shift(6).over("town")
            * 100
        ).alias("town_price_trend"),
    )

    result = result.join(
        town_stats.select(
            "town",
            "transaction_date",
            "town_median_price",
            "town_transaction_volume",
            "town_price_trend",
        ),
        on=["town", "transaction_date"],
        how="left",
    )
    return result


# ════════════════════════════════════════════════════════════════════════
# POINT-IN-TIME RETRIEVAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def as_of(
    df: pl.DataFrame, cutoff: datetime, date_col: str = "transaction_date"
) -> pl.DataFrame:
    """Return rows strictly before ``cutoff`` — the Polars-only PIT path.

    When FeatureStore is unavailable, every technique falls back to this
    helper so the leakage-prevention lesson still runs end-to-end.
    """
    return df.filter(pl.col(date_col) < pl.lit(cutoff.date()))


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH OLS HELPERS — reused across techniques 3 and 4
# ════════════════════════════════════════════════════════════════════════

FEATURE_LIST: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
    "town_median_price",
    "town_price_trend",
]


def prepare_design_matrix(
    df: pl.DataFrame,
    feature_list: list[str] = FEATURE_LIST,
    target: str = "resale_price",
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, build ``[1, X]`` design matrix, return ``(X, y, names)``."""
    model_data = df.drop_nulls(subset=[*feature_list, target])
    X_raw = model_data.select(feature_list).to_numpy().astype(np.float64)
    y = model_data[target].to_numpy().astype(np.float64)
    X = np.column_stack([np.ones(len(y)), X_raw])
    names = ["intercept", *feature_list]
    return X, y, names


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS from scratch and return a dict with betas, SEs, t, p, R²."""
    from scipy import stats as sp_stats  # local import — optional at module load

    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_hat = X @ beta
    resid = y - y_hat

    ssr = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ssr / sst
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - k)
    rmse = float(np.sqrt(ssr / n))

    sigma_sq = ssr / (n - k)
    xtx_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(sigma_sq * np.diag(xtx_inv))
    t_stat = beta / se
    p_val = 2.0 * (1.0 - sp_stats.t.cdf(np.abs(t_stat), df=n - k))

    sse = float(np.sum((y_hat - y.mean()) ** 2))
    f_stat = (sse / (k - 1)) / (ssr / (n - k))
    f_p = 1.0 - sp_stats.f.cdf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "n": n,
        "k": k,
        "beta": beta,
        "se": se,
        "t": t_stat,
        "p": p_val,
        "y_hat": y_hat,
        "resid": resid,
        "r2": float(r2),
        "adj_r2": float(adj_r2),
        "rmse": rmse,
        "f_stat": float(f_stat),
        "f_p": float(f_p),
    }


def normal_normal_posterior(
    beta_hat: float,
    se_hat: float,
    mu_prior: float = 0.0,
    sigma_prior: float = 10_000.0,
) -> dict[str, float]:
    """Normal-Normal conjugate posterior for a single OLS coefficient."""
    prec_prior = 1.0 / sigma_prior**2
    prec_data = 1.0 / se_hat**2
    prec_post = prec_prior + prec_data
    mu_post = (mu_prior * prec_prior + beta_hat * prec_data) / prec_post
    sigma_post = float(np.sqrt(1.0 / prec_post))
    return {
        "mu_post": float(mu_post),
        "sigma_post": sigma_post,
        "ci_low": float(mu_post - 1.96 * sigma_post),
        "ci_high": float(mu_post + 1.96 * sigma_post),
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE SCHEMA BUILDERS — kailash-ml FeatureSchema / FeatureField
# ════════════════════════════════════════════════════════════════════════


def build_schema_v1() -> Any:
    """Return the FeatureSchema v1 definition (basic property features)."""
    from kailash_ml.types import FeatureField, FeatureSchema

    return FeatureSchema(
        name="hdb_property_features",
        features=[
            FeatureField(
                name="floor_area_sqm",
                dtype="float64",
                nullable=False,
                description="Floor area in square metres",
            ),
            FeatureField(
                name="remaining_lease_years",
                dtype="float64",
                nullable=False,
                description="Remaining lease in years",
            ),
            FeatureField(
                name="storey_midpoint",
                dtype="float64",
                nullable=False,
                description="Midpoint of storey range",
            ),
            FeatureField(
                name="price_per_sqm",
                dtype="float64",
                nullable=False,
                description="Transaction price per square metre",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=1,
    )


def build_schema_v2() -> Any:
    """Return FeatureSchema v2 = v1 + three rolling market-context fields."""
    from kailash_ml.types import FeatureField, FeatureSchema

    v1 = build_schema_v1()
    return FeatureSchema(
        name="hdb_property_features",
        features=[
            *v1.features,
            FeatureField(
                name="town_median_price",
                dtype="float64",
                nullable=True,
                description="Median price in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_transaction_volume",
                dtype="int64",
                nullable=True,
                description="Transaction count in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_price_trend",
                dtype="float64",
                nullable=True,
                description="6-month price change % in town",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=2,
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 8.2: Point-in-Time Retrieval — Leakage Prevention
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Retrieve features at specific points in time to prevent leakage
#   - Demonstrate how future data inflates model performance
#   - Compare FeatureStore PIT retrieval with Polars temporal filtering
#   - Quantify the impact of leakage on regression coefficients
#   - Apply PIT correctness to Singapore property market forecasting
#
# PREREQUISITES: Exercise 8.1 (FeatureSchema v1, feature computation)
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — what data leakage is and why it destroys models
#   2. Build — PIT retrieval via FeatureStore and Polars fallback
#   3. Train — compare leaked vs correct model performance
#   4. Visualise — side-by-side leaked vs correct predictions
#   5. Apply — mortgage approval model for DBS Bank Singapore
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from datetime import datetime

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — What Data Leakage Is and Why It Destroys Models


In [ ]:
# Data leakage occurs when information from the future (or from the test
# set) bleeds into the training data. The model learns patterns that
# will not exist at prediction time, producing artificially high
# validation metrics and catastrophic production failures.
#
# Three common leakage types:
#
#   1. TEMPORAL LEAKAGE — Using 2024 transaction prices to train a model
#      that predicts 2023 prices. The model "knows" the future.
#
#   2. TARGET LEAKAGE — Using a feature derived from the target variable.
#      Example: using "price_per_sqm" (which includes resale_price) to
#      predict resale_price. Circular reasoning.
#
#   3. TRAIN-TEST CONTAMINATION — Same transaction appears in both
#      training and test sets (duplication or random split ignoring time).
#
# Point-in-time (PIT) retrieval prevents temporal leakage by enforcing
# a hard cutoff: at prediction time T, only data from before T is used.
#
# Singapore analogy: URA publishes quarterly property price indices.
# If you build a Q1-2024 forecast using data up to Q3-2024, your
# "forecast" is just reading the answer sheet. PIT retrieval is the
# discipline of covering the answer sheet during the exam.



## TASK 2 — BUILD: Point-in-Time retrieval via FeatureStore and Polars


In [ ]:
print("\n" + "=" * 70)
print("  Exercise 8.2 — Point-in-Time Retrieval: Leakage Prevention")
print("=" * 70)

# --- 2a. Prepare features ---
hdb = load_hdb_resale()
features_v1 = compute_v1_features(hdb)
property_schema_v1 = build_schema_v1()

print(f"\n  Features computed: {features_v1.shape[0]:,} rows")
print(
    f"  Date range: {features_v1['transaction_date'].min()} to "
    f"{features_v1['transaction_date'].max()}"
)

# --- 2b. FeatureStore PIT retrieval ---
factory, fs, tracker, has_backend  = await setup_feature_store()

CUTOFF_2023 = datetime(2023, 1, 1)
CUTOFF_2024 = datetime(2024, 1, 1)

if has_backend:
    try:

        async def pit_demo():
            await fs.register_features(property_schema_v1)
            await fs.store(features_v1, property_schema_v1)
            f_2023 = await fs.get_training_set(
                schema=property_schema_v1,
                start=datetime(2000, 1, 1),
                end=CUTOFF_2023,
            )
            f_2024 = await fs.get_training_set(
                schema=property_schema_v1,
                start=datetime(2000, 1, 1),
                end=CUTOFF_2024,
            )
            return f_2023, f_2024

        features_2023, features_2024  = await pit_demo()
        delta = features_2024.height - features_2023.height
        print(f"\n  [FeatureStore PIT]")
        print(f"    Features as of 2023-01-01: {features_2023.height:,} rows")
        print(f"    Features as of 2024-01-01: {features_2024.height:,} rows")
        print(f"    2023 transactions added: {delta:,}")
    except Exception as e:
        has_backend = False
        print(f"  [Skipped: PIT retrieval ({type(e).__name__}: {e})]")

if not has_backend:
    # TODO: Use the Polars PIT fallback to filter features before each cutoff.
    # Hint: as_of(features_v1, CUTOFF_2023) returns rows strictly before
    # the cutoff date. Do the same for CUTOFF_2024.
    features_2023 = ____
    features_2024 = ____
    delta = features_2024.height - features_2023.height
    print(f"\n  [Polars PIT]")
    print(f"    Features before 2023: {features_2023.height:,}")
    print(f"    Features before 2024: {features_2024.height:,}")
    print(f"    Delta: {delta:,}")

print(f"\n  --- Why Point-in-Time Matters ---")
print(f"  To predict prices at T=2023-01-01, you must ONLY use data before T.")
print(f"  Using 2024 data would leak future info -> over-optimistic evaluation.")



### Checkpoint 1


In [ ]:
assert features_2023.height > 0, "Task 2: must have pre-2023 features"
assert (
    features_2024.height > features_2023.height
), "Task 2: 2024 cutoff must include more rows than 2023"
print("\n[ok] Checkpoint 1 passed — PIT retrieval demonstrated\n")

# INTERPRETATION: The delta between 2023 and 2024 cutoffs represents
# an entire year of transactions that would LEAK into a 2023 model
# if we used a naive "use all data" approach. In production, this
# means the model would appear to predict perfectly for 2023 but fail
# on genuinely future data.



## TASK 3 — TRAIN: Compare leaked vs PIT-correct model performance


In [ ]:
# We build two OLS models:
#   - CORRECT: trained on data before 2023, evaluated on 2023 data
#   - LEAKED:  trained on ALL data including 2023, evaluated on 2023
#
# The leaked model will show inflated R² because it already "knows"
# the 2023 prices it's being asked to predict.

print("--- Comparing Leaked vs Correct Models ---")

# Correct model: train on pre-2023, test on 2023
FEATURE_COLS = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
]

train_correct = features_2023.drop_nulls(subset=[*FEATURE_COLS, "resale_price"])

# TODO: Filter features_v1 to get ONLY 2023 transactions (>= 2023-01-01
# and < 2024-01-01), then drop nulls. This is the test set.
# Hint: use pl.col("transaction_date") >= pl.lit(CUTOFF_2023.date())
# and < pl.lit(CUTOFF_2024.date())
test_2023 = ____

# Build design matrices
X_train = np.column_stack(
    [
        np.ones(train_correct.height),
        train_correct.select(FEATURE_COLS).to_numpy().astype(np.float64),
    ]
)
y_train = train_correct["resale_price"].to_numpy().astype(np.float64)

X_test = np.column_stack(
    [
        np.ones(test_2023.height),
        test_2023.select(FEATURE_COLS).to_numpy().astype(np.float64),
    ]
)
y_test = test_2023["resale_price"].to_numpy().astype(np.float64)

# TODO: Fit the CORRECT model using fit_ols on the training data.
# Hint: fit_ols(X_train, y_train) returns a dict with "beta", "r2", etc.
ols_correct = ____

y_pred_correct = X_test @ ols_correct["beta"]
resid_correct = y_test - y_pred_correct
rmse_correct = float(np.sqrt(np.mean(resid_correct**2)))
ss_res_correct = float(np.sum(resid_correct**2))
ss_tot = float(np.sum((y_test - y_test.mean()) ** 2))
r2_test_correct = 1 - ss_res_correct / ss_tot

# Leaked model: train on ALL data (including 2023 test set)
X_all = np.column_stack(
    [
        np.ones(features_v1.height),
        features_v1.drop_nulls(subset=[*FEATURE_COLS, "resale_price"])
        .select(FEATURE_COLS)
        .to_numpy()
        .astype(np.float64),
    ]
)
y_all = (
    features_v1.drop_nulls(subset=[*FEATURE_COLS, "resale_price"])["resale_price"]
    .to_numpy()
    .astype(np.float64)
)

# TODO: Fit the LEAKED model using fit_ols on ALL data.
# Hint: fit_ols(X_all, y_all)
ols_leaked = ____

y_pred_leaked = X_test @ ols_leaked["beta"]
resid_leaked = y_test - y_pred_leaked
rmse_leaked = float(np.sqrt(np.mean(resid_leaked**2)))
ss_res_leaked = float(np.sum(resid_leaked**2))
r2_test_leaked = 1 - ss_res_leaked / ss_tot

print(f"\n  Correct model (trained pre-2023, tested on 2023):")
print(f"    Training R²: {ols_correct['r2']:.4f}")
print(f"    Test RMSE:   ${rmse_correct:,.0f}")
print(f"    Test R²:     {r2_test_correct:.4f}")

print(f"\n  Leaked model (trained on ALL data, tested on 2023):")
print(f"    Training R²: {ols_leaked['r2']:.4f}")
print(f"    Test RMSE:   ${rmse_leaked:,.0f}")
print(f"    Test R²:     {r2_test_leaked:.4f}")

leakage_gap = rmse_correct - rmse_leaked
print(f"\n  Leakage gap: ${leakage_gap:,.0f} RMSE difference")
print(f"  The leaked model APPEARS ${abs(leakage_gap):,.0f} better per prediction")
print(f"  but this improvement is fictional — it used future data.")



### Checkpoint 2


In [ ]:
assert ols_correct["r2"] > 0.1, "Task 3: correct model R² should be reasonable"
assert rmse_correct > 0, "Task 3: RMSE must be positive"
print("\n[ok] Checkpoint 2 passed — leaked vs correct comparison complete\n")

# INTERPRETATION: Even a small RMSE gap from leakage is dangerous.
# In production, the leaked model's confidence intervals are too
# narrow (it thinks it knows more than it does), and every decision
# based on those intervals is over-confident.



## TASK 4 — VISUALISE: Side-by-side leaked vs correct predictions


In [ ]:
print("--- Visualising Leakage Impact ---")

rng = np.random.default_rng(42)
n_sample = min(2000, len(y_test))
idx = rng.choice(len(y_test), size=n_sample, replace=False)

# TODO: Create a side-by-side subplot comparing correct vs leaked.
# Hint: make_subplots(rows=1, cols=2, subplot_titles=[...])
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"CORRECT (PIT) — RMSE ${rmse_correct:,.0f}",
        f"LEAKED — RMSE ${rmse_leaked:,.0f}",
    ],
)

# Correct model
fig.add_trace(
    go.Scatter(
        x=y_test[idx].tolist(),
        y=y_pred_correct[idx].tolist(),
        mode="markers",
        marker={"size": 3, "opacity": 0.4, "color": "steelblue"},
        name="Correct",
    ),
    row=1,
    col=1,
)
# Leaked model
fig.add_trace(
    go.Scatter(
        x=y_test[idx].tolist(),
        y=y_pred_leaked[idx].tolist(),
        mode="markers",
        marker={"size": 3, "opacity": 0.4, "color": "firebrick"},
        name="Leaked",
    ),
    row=1,
    col=2,
)

# Perfect prediction lines
for col in [1, 2]:
    fig.add_trace(
        go.Scatter(
            x=[float(y_test.min()), float(y_test.max())],
            y=[float(y_test.min()), float(y_test.max())],
            mode="lines",
            line={"dash": "dash", "color": "gray"},
            showlegend=False,
        ),
        row=1,
        col=col,
    )

fig.update_layout(
    title="Data Leakage Impact — Actual vs Predicted (2023 Test Set)",
    height=500,
    width=1000,
)
fig.update_xaxes(title_text="Actual Price ($)", row=1, col=1)
fig.update_xaxes(title_text="Actual Price ($)", row=1, col=2)
fig.update_yaxes(title_text="Predicted Price ($)", row=1, col=1)

fig.write_html(str(OUTPUT_DIR / "02_leakage_comparison.html"))
print(f"\n  Saved: {OUTPUT_DIR / '02_leakage_comparison.html'}")

# Residual comparison
fig2 = go.Figure()
fig2.add_trace(
    go.Histogram(
        x=resid_correct[idx].tolist(),
        name=f"Correct (RMSE ${rmse_correct:,.0f})",
        opacity=0.6,
        nbinsx=50,
    )
)
fig2.add_trace(
    go.Histogram(
        x=resid_leaked[idx].tolist(),
        name=f"Leaked (RMSE ${rmse_leaked:,.0f})",
        opacity=0.6,
        nbinsx=50,
    )
)
fig2.update_layout(
    title="Residual Distributions — Correct vs Leaked",
    xaxis_title="Residual ($)",
    yaxis_title="Count",
    barmode="overlay",
)
fig2.write_html(str(OUTPUT_DIR / "02_residual_comparison.html"))
print(f"  Saved: {OUTPUT_DIR / '02_residual_comparison.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — leakage visualisations saved\n")



## TASK 5 — APPLY: Mortgage Approval Model for DBS Bank Singapore


In [ ]:
# Scenario: DBS Bank builds a mortgage risk model using HDB resale
# prices. The model estimates "loan-to-value" (LTV) ratios to decide
# mortgage approval. If the model leaks future prices, it
# overestimates property values, approves mortgages for overvalued
# flats, and the bank absorbs losses when prices correct.
#
# DBS processes ~8,000 HDB mortgages per month in Singapore. Average
# mortgage: S$350,000. If future-price leakage inflates valuations
# by 5%, the bank over-lends by ~$17,500 per mortgage.
#
#   Monthly exposure: 8,000 * $17,500 = S$140M in excess lending
#   If 3% of these correct downward: S$4.2M monthly write-offs
#   Annual impact: ~S$50M
#
# PIT retrieval eliminates this by ensuring the valuation model only
# uses prices that existed BEFORE the mortgage application date.

print("=== APPLY: DBS Mortgage Approval — PIT-Correct Valuation ===")
print()
print("  Scenario: DBS Bank HDB mortgage risk model")
print()
print("  WITHOUT PIT retrieval:")
print("    - Model trained on all data, including future prices")
print("    - Overestimates property values by ~5%")
print("    - Over-lends S$17,500 per mortgage on average")
print("    - 8,000 mortgages/month * 3% correction rate")
print("    - Annual write-off exposure: ~S$50M")
print()
print("  WITH PIT retrieval:")
print("    - Model only uses prices before application date")
print("    - Conservative valuations that reflect actual market")
print("    - No systematic over-lending from future data")
print()
print(f"  Your PIT model performance on 2023 data:")
print(f"    R² = {r2_test_correct:.4f} (honest, no leakage)")
print(f"    RMSE = ${rmse_correct:,.0f} (true prediction error)")
print(f"    vs leaked RMSE = ${rmse_leaked:,.0f} (fictionally low)")



### Checkpoint 4


In [ ]:
print("\n[ok] Checkpoint 4 passed — PIT mortgage application demonstrated\n")



## REFLECTION


[ok] Point-in-time retrieval: hard temporal cutoffs for training data
  [ok] Leakage detection: comparing leaked vs correct model performance
  [ok] FeatureStore PIT API: get_training_set(start, end) enforcement
  [ok] Polars temporal filtering: as_of() fallback for Polars-only mode
  [ok] Quantified leakage impact: RMSE gap and dollar-value consequences

  KEY INSIGHT: A model that "works great in development" but fails in
  production almost always has a leakage bug. PIT retrieval is the
  structural fix — it makes leakage impossible, not just unlikely.

  Next: In 03_rolling_features.py, you'll extend the feature schema
  with rolling market statistics that capture town-level price trends
  and transaction volumes over time.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

